In [28]:
using LinearAlgebra
using Plots
using DataStructures

## Rotational and Radial Kinetic Energy

$$\alpha,\alpha^\prime \in \{1, \dots N\} \space | \space \beta,\beta^\prime \in [0, 2 \pi]$$

Off Diagonal

$$T_{r \alpha \alpha^{\prime}} = \left(\frac{\hbar^2}{2\mu \Delta r^2 }\right) \cdot \left (-1\right)^{\left (\alpha - \alpha^{\prime} \right)} \cdot \left(\frac{\pi^2}{3} - \frac{1}{2\alpha^2}\right) $$

$${T}_{\phi \beta \beta^{\prime}} = \left(\frac{\hbar^2}{2\mu}\right) \cdot \left (-1\right)^{\left (\beta - \beta^{\prime} \right)} \cdot \left(\frac{N(N+1)}{3} \right)$$

On Diagonal 

$$T_{r \alpha \alpha^{\prime}} = \left(\frac{\hbar^2}{2\mu \Delta r^2}\right) \cdot \left(\frac{2}{(\alpha-\alpha^{\prime})^2} - \frac{2}{(\alpha+\alpha^{\prime})^2} \right)$$

$${T}_{\phi \beta \beta^{\prime}} = \left(\frac{\hbar^2}{2\mu}\right) \cdot \frac{cos\left(\frac{\pi(\beta-\beta)}{2N+1}\right)}{2sin\left(\frac{\pi(\beta-\beta)}{2N+1}\right)^2}$$



## DVR functions

In [29]:
function dvr_rotational(N, mu)
    h_bar = 1
    phi_grid = zeros((N))
    T = zeros((N, N))
    for a=1:N
        phi_grid[a] = (2*pi*(a-1)/(N))
        for ap=1:N
            if a==ap # On-diagonal
                T[a,ap] = h_bar^2 / 2*mu * (-1)^(a-ap) * (N*(N+1)/3)
            else # Off-Diagonal
                T[a,ap] = h_bar^2 / 2*mu * (-1)^(a-ap) * (cos(pi*(a-ap)/(2*N+1)))/(2*sin(pi*(a-ap)/(2*N+1))^2)
            
            end
        end
    end
    return T, phi_grid
end

function dvr_radial(N, r_min, r_max, mu)
    h_bar = 1
    r_grid = zeros((N))
    T = zeros((N, N))
    d_r = (r_max-r_min)/(N-1)
    for i=1:N
        r_grid[i] = d_r*(i-1) + r_min
    end
    for b=1:N
        for bp=1:N
            if b==bp # On-diagonal 
                T[b,bp] = h_bar^2 / (2*mu*d_r^2) * (-1)^(b-bp) * (pi^2/3 - 1/(2*b^2))
            else # Off-diagonal
                T[b,bp] = h_bar^2 / (2*mu*d_r^2) * (-1)^(b-bp) * (2/(b-bp)^2 - 2/(b+bp)^2)
            end
        end
    end
    return T, r_grid
end

dvr_radial (generic function with 1 method)

## Harmonic and Dipole-Dipole Potential

In $(r, \phi)$ basis

$$V_{\alpha} = \frac{1}{2} k \left(r_{\alpha} - r_e \right)^2$$

$$\hat{V}_{1,2} = \frac{\left(\frac{\mu}{r_e} \right)^2}{4 \pi \epsilon_o R^3} \cdot \hat{r}_1 \hat{r}_2 \cdot \left(sin (\hat{\phi}_1) sin (\hat{\phi}_2) - 2 cos (\hat{\phi}_1) cos (\hat{\phi}_2) \right)$$

In [30]:
function V_oscilation(N, k, r_e, r_grid)

    V = zeros((N))

    for i=1:N
        V[i] = 0.5*k * (r_grid[i] - r_e)^2
    end
    return V
end

function V_neighbours(N_r, N_phi, mu, r_grid, phi_grid, r_e, R_3)
    epsilon_o = 1
    size = N_r*N_phi
    V = zeros((size, size))
    i = 0
    C = ((mu/r_e)^2)/(4*pi*epsilon_o*R_3^3)
    C = 0
    for a1=1:N_r
        for a2=1:N_r
            i+=1
            j=0
            temp = C * r_grid[a1]*r_grid[a2]
            for b1=1:N_phi
                for b2=1:N_phi
                    j+=1
                    V[i,j] = temp * sin(phi_grid[b1])*sin(phi_grid[b2]) - 2*cos(phi_grid[b1])*cos(phi_grid[b2])
                end
            end
        end
    end
    return V
end

V_neighbours (generic function with 1 method)

## Constants for HF molecule

In [31]:
amu_to_au = 1/0.00054858 #au
ang_to_bohr = 1/0.529177 #1/bohr radius

wavenumber_to_Hartrees = 0.00000455633

m_H = 1.008 #in amu
m_F = 18.998 #in amu

r_e = 0.9168 * ang_to_bohr #in Angstrom

omega_e = 4138.385 #Hartrees

m_H *= amu_to_au
m_F *= amu_to_au

mu = (m_H*m_F)/(m_H+m_F) #reduced mass

omega_e *= wavenumber_to_Hartrees #h_bar = 1 (in au)

k = omega_e^2 * mu

Alpha = (mu*omega_e)/2 # h_bar not included as h_bar = 1




16.450694885570016

$$e^{(-\frac{\mu \omega r^2}{2 \hbar})}$$

For gaussian Distribution of vibrational states

$$\alpha (r-r_e)^2 = order$$

$$r = r_e \pm \sqrt{\frac{order}{\alpha}}

In [32]:
order = 12

r_min = r_e - sqrt(order/Alpha)
r_max = r_e + sqrt(order/Alpha)

2.5865814976519044

In [33]:
N_r = 24
N_phi = 16

mmax = 10

size = N_r * N_phi

R = 10 #Distance between rotors

10

In [34]:
T_r, r_grid = dvr_radial(N_r, r_min, r_max, mu)

T_phi, phi_grid = dvr_rotational(N_phi, mu)

V_r = V_oscilation(N_r, k, r_e, r_grid)

24-element Vector{Float64}:
 0.22627017272460007
 0.18862976591975167
 0.15441121427898044
 0.12361451780228623
 0.09623967648966919
 0.07228669034112935
 0.051755559356666564
 0.03464628353628091
 0.020958862879972427
 0.010693297387741057
 0.0038495870595867837
 0.0004277318955096443
 0.00042773189550963405
 0.003849587059586753
 0.010693297387741005
 0.020958862879972354
 0.03464628353628082
 0.05175555935666651
 0.07228669034112921
 0.09623967648966904
 0.12361451780228615
 0.15441121427898022
 0.1886297659197514
 0.22627017272459993

## 1-body Hamiltonian, in the $(\alpha, m)$ basis

In [35]:
function Hamiltonian_1body(N_r, T_r, V_r, mu, m)
    H = zeros((N_r, N_r))
    for a=1:N_r
        H[a,a] = V_r[a] + 1/(2*mu*r_grid[a]^2) * m^2
        for ap=1:N_r
            H[a,ap] += T_r[a,ap]
        end
    end
    return H
end

Hamiltonian_1body (generic function with 1 method)

index for $\langle n_m | \alpha m \rangle$ and $\langle \alpha m | n_m \rangle$

For $| \alpha m \rangle = m \times $ mmax $ + \alpha $



In [36]:
mmax=10

lst = []

h_m = Hamiltonian_1body(N_r, T_r, V_r, mu, 0)
h_m_1up = Hamiltonian_1body(N_r, T_r, V_r, mu, 1)

evec_n_m_1UP = eigvecs(h_m_1up)
evec_n_m = eigvecs(h_m)
evec_n_m_1UP = evec_n_m_1UP

for m=0:mmax
    if m > 0
        h_m_1up = Hamiltonian_1body(N_r, T_r, V_r, mu, m+1)
        
        evec_n_m_1down = evec_n_m
        evec_n_m = evec_n_m_1UP
        evec_n_m_1UP = eigvecs(h_m_1up)        
    end
    
    for n=1:2
        lst_m = []
        for a=1:N_r
            val = 0
            
            idx_m = n*m + a
            idx_m_down = n*(m-1) + a
            idx_m_up = n*(m+1) + a
            
            temp = 0
            if m > 0
                temp = 1/2 * (evec_n_m_1down[idx_m_down] + evec_n_m_1UP[idx_m_up])
            else
                temp = evec_n_m_1UP[idx_m_up]
            end

            val = evec_n_m[idx_m] * r_grid[a] * temp
            push!(lst_m, val)
        end
        push!(lst, lst_m)
    end
    

end

lst_matrix = reduce(hcat, lst)

println("For energy levels, m, from 0 to $(mmax), alpha_max = $N_r")

lst_matrix


For energy levels, m, from 0 to 10, alpha_max = 24


24×22 Matrix{Any}:
 5.74375e-11  3.53116e-10  1.31207e-9   …   0.117451      5.2811e-6
 2.857e-9     1.46436e-8   4.54124e-8       0.193596      3.30052e-7
 9.69402e-8   4.14364e-7   1.08988e-6       0.226032      1.41075e-8
 2.27525e-6   8.1107e-6    1.81929e-5       0.187315      2.71606e-10
 3.69751e-5   0.000109924  0.000211832      0.110176     -1.85648e-9
 0.000416289  0.00103215   0.00172626   …   0.0459138    -8.46443e-8
 0.00324856   0.00671741   0.00988471       0.0135169    -2.33175e-6
 0.017578     0.0303143    0.0399495        0.00280171   -4.23161e-5
 0.0659746    0.0948907    0.114508         0.000407524  -0.000504206
 0.171804     0.206088     0.233853         4.14773e-5   -0.00389956
 0.310488     0.310626     0.34157      …   2.94695e-6   -0.0191171
 0.389498     0.324992     0.357614         1.45893e-7   -0.0563103
 0.339229     0.236067     0.26844          5.02125e-9   -0.0850679
 0.205153     0.119069     0.144212         8.48574e-11  -0.0150277
 0.0861628    0.04

Matrix columns, $n_{m=0}=0$, $n_{m=0}=1$, $n_{m=1}=0$, $n_{m=1}=1, \dots n_{m={mmax}}=0, n_{m={mmax}}=1$

Row count $= N_r$